In [2]:
import pandas as pd
import numpy as np

df_t = pd.read_csv('../data/transactions.csv')
df_e = pd.read_csv('../data/employes.csv')

print(f"Transactions : {df_t.shape}")
print(f"Employés     : {df_e.shape}")

print("=== TRANSACTIONS — types ===")
print(df_t.dtypes)
print("\nValeurs nulles :")
print(df_t.isnull().sum())

print("\n=== EMPLOYES — types ===")
print(df_e.dtypes)
print("\nValeurs nulles :")
print(df_e.isnull().sum())

df_t_clean = df_t.copy()

# Suppression des doublons
nb_dup = df_t_clean.duplicated().sum()
df_t_clean = df_t_clean.drop_duplicates()
print(f"Doublons supprimés : {nb_dup}")

# Suppression des montants négatifs ou nuls
nb_neg = (df_t_clean['amount'] <= 0).sum()
df_t_clean = df_t_clean[df_t_clean['amount'] > 0]
print(f"Montants invalides supprimés : {nb_neg}")

# Renommage des colonnes pour plus de clarté
df_t_clean = df_t_clean.rename(columns={
    'nameOrig'       : 'compte_source',
    'nameDest'       : 'compte_dest',
    'oldbalanceOrg'  : 'solde_avant_source',
    'newbalanceOrig' : 'solde_apres_source',
    'oldbalanceDest' : 'solde_avant_dest',
    'newbalanceDest' : 'solde_apres_dest',
    'isFraud'        : 'est_fraude',
    'isFlaggedFraud' : 'est_flagge'
})

print(f"\nTransactions après nettoyage : {df_t_clean.shape}")
print(df_t_clean.head(3))

df_e_clean = df_e.copy()

# Suppression des doublons
nb_dup = df_e_clean.duplicated().sum()
df_e_clean = df_e_clean.drop_duplicates()
print(f"Doublons supprimés : {nb_dup}")

# Conversion des colonnes de dates
date_cols = ['recorddate_key', 'birthdate_key', 'orighiredate_key', 'terminationdate_key']
for col in date_cols:
    df_e_clean[col] = pd.to_datetime(df_e_clean[col], errors='coerce')

# Standardisation de la colonne STATUS en majuscules
df_e_clean['STATUS'] = df_e_clean['STATUS'].str.upper().str.strip()

# Renommage des colonnes
df_e_clean = df_e_clean.rename(columns={
    'EmployeeID'          : 'employe_id',
    'recorddate_key'      : 'date_enregistrement',
    'birthdate_key'       : 'date_naissance',
    'orighiredate_key'    : 'date_embauche',
    'terminationdate_key' : 'date_depart',
    'age'                 : 'age',
    'length_of_service'   : 'anciennete',
    'city_name'           : 'ville',
    'department_name'     : 'departement',
    'job_title'           : 'poste',
    'store_name'          : 'magasin',
    'gender_full'         : 'genre',
    'termreason_desc'     : 'raison_depart',
    'termtype_desc'       : 'type_depart',
    'STATUS_YEAR'         : 'annee',
    'STATUS'              : 'statut',
    'BUSINESS_UNIT'       : 'unite'
})

# On garde uniquement les colonnes utiles
df_e_clean = df_e_clean.drop(columns=['gender_short'])

print(f"\nEmployés après nettoyage : {df_e_clean.shape}")
print(df_e_clean.head(3))


# Rapport qualité synthétique
rapport = pd.DataFrame({
    'dataset'       : ['transactions', 'employes'],
    'lignes_brutes' : [len(df_t), len(df_e)],
    'lignes_clean'  : [len(df_t_clean), len(df_e_clean)],
    'colonnes'      : [len(df_t_clean.columns), len(df_e_clean.columns)],
    'nulls_restants': [df_t_clean.isnull().sum().sum(), df_e_clean.isnull().sum().sum()],
    'doublons_restants' : [df_t_clean.duplicated().sum(), df_e_clean.duplicated().sum()]
})

print(rapport)

# Sauvegarde des fichiers nettoyés
df_t_clean.to_csv('../data/transactions_clean.csv', index=False)
df_e_clean.to_csv('../data/employes_clean.csv', index=False)
rapport.to_csv('../outputs/rapport_qualite.csv', index=False)

print("\nFichiers sauvegardés dans data/ et outputs/")

Transactions : (6362620, 11)
Employés     : (49653, 18)
=== TRANSACTIONS — types ===
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Valeurs nulles :
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

=== EMPLOYES — types ===
EmployeeID             int64
recorddate_key           str
birthdate_key            str
orighiredate_key         str
terminationdate_key      str
age                    int64
length_of_service      int64
city_name                str
department_name          str
job_title                str
store_name             int64
gender_short       